# 02_chroniques : Niveaux des nappes autour de Valence

### **Descriptif du Notebook VERSION VS CODE** :

**Le projet :**
- Connexion à l'API Hub'Eau sur le jeu de données "Piézométrie" (https://hubeau.eaufrance.fr/page/api-piezometrie)
- Données recherchées :
    - A. Notebook `01_stations` : la liste des stations piézométriques de la zone → endpoint `/stations`
    - B. **ICI** : le niveau d'eau mesuré dans chaque station (chroniques) → endpoint `/chroniques`
- Zone géographique : le bassin de Valence (26000) = bbox « carré de 100 km centré sur Valence »

**Entrée :**
- `data/processed/stations_mesurees.csv` : les stations ayant au moins une mesure → **produit par `01_stations`** (à exécuter avant)

**Les étapes du Notebook :**

1. Imports
2. Configuration : variables du projet (.env) + paramètres de la période étudiée
3. Chargement des stations (sortie du notebook 01)
4. Récupération des chroniques : **1 appel à l'API par station**
5. Mise en DataFrame et contrôle qualité
6. Sauvegarde des données brutes (`data/raw/`)
7. Sauvegarde des données traitées (`data/processed/`)

**Sorties :**
- `data/raw/chroniques_AAAA-MM-JJ.json` : données brutes de l'API + contexte de l'extraction
- `data/processed/chroniques_depuis_<DATE_DEBUT>.csv` : mesures nettoyées, format long (1 ligne = 1 station × 1 date)

### Prérequis avant d'utiliser le notebook

**Première installation (une seule fois)**
- Cloner le dépôt, puis dans le terminal, à la racine du projet :
  `python3 -m venv .venv` → `source .venv/bin/activate` → `pip install -r requirements.txt`
- Créer le fichier de variables : `cp .env.example .env`

**À chaque session de travail**
- Ouvrir le **dossier racine** `Hub-eau` dans VS Code (pas seulement le notebook)
- Récupérer les dernières modifications : `git pull`
- Vérifier que le **kernel** sélectionné (en haut à droite) est `.venv`
- Exécuter la cellule de vérification ci-dessous

**Pour installer une nouvelle bibliothèque**
- Activer le venv dans le terminal : `source .venv/bin/activate` (l'invite doit commencer par `(.venv)`)
- `pip install <bibliothèque>`, puis `pip freeze > requirements.txt`, puis commit

**Avant chaque commit**
- *Restart Kernel* + *Run All* : le notebook doit tourner en entier sans erreur
- `git status` : `.env`, `.venv/` et les fichiers de `data/` ne doivent jamais apparaître

In [1]:
# Vérification de l'environnement : le kernel doit être celui du .venv du projet
import sys

print("Python utilisé :", sys.executable)
assert ".venv" in sys.executable, "⚠️ Mauvais kernel : sélectionner le .venv en haut à droite"

Python utilisé : /home/virginie/Documents/DATA/00-PROJETS/Hub-eau/.venv/bin/python


## **Appel API - phase B /chroniques**

In [2]:
# 1. Imports des bibliothèques

# Rechargement automatique des modules de src/ quand ils sont modifiés
%load_ext autoreload
%autoreload 2

# Bibliothèque standard (livrée avec Python)
import json                         # sauvegarde des données brutes
import os                           # lecture des variables d'environnement
import sys                          # chemin de recherche des modules
import time                         # pause entre deux stations
from datetime import datetime       # horodatage des fichiers sauvegardés
from pathlib import Path            # chemins de fichiers (data/raw/...)

# Bibliothèques externes (installées dans le .venv, listées dans requirements.txt)
import pandas as pd                 # mise en DataFrame et contrôle des données
import requests                     # uniquement pour intercepter les erreurs réseau (requests.RequestException)
from dotenv import load_dotenv      # chargement du fichier .env

# Modules locaux (dossier src/ à la racine du projet)
sys.path.append(str(Path.cwd().parent))     # rend la racine du projet visible pour Python
from src.hubeau import appeler_api

In [3]:
# 2. Configuration

# --- A. Variables d'environnement (fichier .env) ---
load_dotenv()                                   # charge le .env (cherché en remontant les dossiers)

BASE_URL = os.getenv("HUBEAU_BASE_URL")
if BASE_URL is None:
    raise ValueError("HUBEAU_BASE_URL introuvable : vérifier que le fichier .env existe à la racine du projet")

URL_CHRONIQUES = f"{BASE_URL}/chroniques"       # mesures de niveau d'eau par station

# --- B. Période étudiée (choix d'analyse) ---
DATE_DEBUT_CHRONIQUES = "2020-01-01"            # début de la période (AAAA-MM-JJ) ; élargir pour plus d'historique

# --- C. Paramètres techniques de l'API ---
PAGE_SIZE_CHRONIQUES = 20000                    # max autorisé par l'API (page * size <= 20000)
TIMEOUT_S = 30                                  # délai max d'attente d'une réponse (secondes)
PAUSE_S = 0.2                                   # pause entre deux appels (respect de l'API)

# --- D. Chemins du projet ---
PROJECT_ROOT = Path.cwd().parent                # le notebook est dans notebooks/ => on remonte d'un niveau
assert (PROJECT_ROOT / "requirements.txt").exists(), f"Racine du projet incorrecte : {PROJECT_ROOT}"

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_RAW.mkdir(parents=True, exist_ok=True)     # créé s'il n'existe pas
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

FICHIER_STATIONS = DATA_PROCESSED / "stations_mesurees.csv"   # entrée : produit par 01_stations

# --- Récapitulatif ---
print("URL chroniques   :", URL_CHRONIQUES)
print("Période          : depuis le", DATE_DEBUT_CHRONIQUES)
print("Racine projet    :", PROJECT_ROOT)
print("Fichier stations :", FICHIER_STATIONS)

URL chroniques   : https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/chroniques
Période          : depuis le 2020-01-01
Racine projet    : /home/virginie/Documents/DATA/00-PROJETS/Hub-eau
Fichier stations : /home/virginie/Documents/DATA/00-PROJETS/Hub-eau/data/processed/stations_mesurees.csv


#### 3. Chargement des stations

On ne rappelle pas l'API `/stations` : on lit le fichier produit par le notebook `01_stations`.
Le CSV ne conserve pas les types → les colonnes de dates sont reconverties avec `parse_dates`.

In [4]:
# 3. Chargement des stations avec mesures (sortie du notebook 01_stations)

if not FICHIER_STATIONS.exists():
    raise FileNotFoundError(f"{FICHIER_STATIONS} introuvable : exécuter d'abord le notebook 01_stations")

df_stations_mesurees = pd.read_csv(FICHIER_STATIONS,
                                   parse_dates=["date_debut_mesure", "date_fin_mesure"])

codes_stations = df_stations_mesurees["code_bss"].tolist()

print(f"{len(codes_stations)} stations à interroger")
df_stations_mesurees[["code_bss", "nom_commune", "date_debut_mesure",
                      "date_fin_mesure", "nb_mesures_piezo"]].head()

60 stations à interroger


,code_bss,nom_commune,date_debut_mesure,date_fin_mesure,nb_mesures_piezo
0,08422X0190/F1,Loriol-sur-Drôme,1999-12-17,2026-11-05,9149
1,07948X0047/F1,Romans-sur-Isère,2009-06-12,2026-11-06,6129
2,08664X0069/F1,Soyans,2008-12-03,2026-11-04,6519
3,07703X0064/P1,Anneyron,2009-06-09,2026-04-20,6155
4,08187X0238/P1,Beaumont-lès-Valence,2009-06-26,2026-11-06,6336


In [5]:
# Stations sans mesure sur la période étudiée (arrêtées avant DATE_DEBUT_CHRONIQUES)
# => l'API renverra 0 résultat pour elles : c'est attendu, pas une erreur

arretees = df_stations_mesurees[df_stations_mesurees["date_fin_mesure"] < DATE_DEBUT_CHRONIQUES]
print(f"{len(arretees)} station(s) sans mesure attendue depuis le {DATE_DEBUT_CHRONIQUES}")
arretees[["code_bss", "nom_commune", "date_fin_mesure"]]

8 station(s) sans mesure attendue depuis le 2020-01-01


,code_bss,nom_commune,date_fin_mesure
23,08188X0054/F,La Baume-Cornillane,2013-03-08
25,07952X0027/F,Châtillon-Saint-Jean,2013-01-11
38,08662X0049/P,Saint-Marcel-lès-Sauzet,2017-03-29
40,07944X0048/S,Margès,1998-04-28
42,07944X0049/S,Margès,2018-07-09
43,08184X0069/S,Valence,1998-08-22
45,08413X0081/LAUZAS,Creysseilles,2002-02-26
46,08666X0161/PUITS,Montboucher-sur-Jabron,2013-01-08


#### 4. Récupération des chroniques : 1 appel par station

L'API limite chaque requête à **20 000 résultats** (`page × size ≤ 20 000`).
Toutes stations confondues, il y a bien plus de mesures : on interroge donc **une station à la fois**
(une station reste sous la limite), et on cumule les résultats dans une seule liste (`extend`).

- `try / except` : une station en échec est notée dans `erreurs`, la boucle continue.
- On n'intercepte que les erreurs attendues (`RuntimeError` = erreur renvoyée par l'API,
  `requests.RequestException` = problème réseau) : une erreur de code doit, elle, arrêter l'exécution.
- **Relance** : les stations en échec (ex. délai dépassé ponctuel) sont retentées une fois,
  avec un délai d'attente plus long.
- Durée : environ 1 minute.

In [6]:
# 4a. Fonction de récupération : boucle sur une liste de stations

def recuperer_chroniques(codes, timeout_s):
    '''Interroge /chroniques pour chaque station. Renvoie (mesures, erreurs).'''
    mesures, echecs = [], []

    for i, code in enumerate(codes, start=1):
        print(f"[{i}/{len(codes)}] {code} : ", end="")

        params_chroniques = {
            "code_bss": code,
            "date_debut_mesure": DATE_DEBUT_CHRONIQUES,
            "size": PAGE_SIZE_CHRONIQUES,
            "format": "json",
        }

        try:
            mesures.extend(appeler_api(URL_CHRONIQUES, params_chroniques,
                                       timeout_s=timeout_s, pause_s=PAUSE_S))
        except (RuntimeError, requests.RequestException) as e:
            echecs.append({"code_bss": code, "erreur": str(e)})
            print(f"⚠️ Échec : {e}")

        time.sleep(PAUSE_S)  # pause entre deux stations

    return mesures, echecs

In [7]:
# 4b. Récupération des chroniques, station par station

chroniques, erreurs = recuperer_chroniques(codes_stations, timeout_s=TIMEOUT_S)

print(f"\n✅ {len(chroniques)} mesures récupérées pour {len(codes_stations)} stations")
print(f"⚠️ {len(erreurs)} station(s) en échec")

[1/60] 08422X0190/F1 : Page 1 : 2505 résultats (2505/2505)
[2/60] 07948X0047/F1 : Page 1 : 2501 résultats (2501/2501)
[3/60] 08664X0069/F1 : Page 1 : 2505 résultats (2505/2505)
[4/60] 07703X0064/P1 : Page 1 : 2297 résultats (2297/2297)
[5/60] 08187X0238/P1 : Page 1 : 2504 résultats (2504/2504)
[6/60] 07955X0099/P1 : Page 1 : 2505 résultats (2505/2505)
[7/60] 08184X0084/PZ1 : Page 1 : 2465 résultats (2465/2465)
[8/60] 08422X0191/F2 : Page 1 : 2503 résultats (2503/2503)
[9/60] 08424X0006/F2 : Page 1 : 2504 résultats (2504/2504)
[10/60] 08426X0133/P2 : ⚠️ Échec : HTTPSConnectionPool(host='hubeau.eaufrance.fr', port=443): Read timed out. (read timeout=30)
[11/60] 07955X0101/PZ2 : Page 1 : 2504 résultats (2504/2504)
[12/60] 07703X0069/F83 : Page 1 : 2364 résultats (2364/2364)
[13/60] 07725X0071/F3 : Page 1 : 2505 résultats (2505/2505)
[14/60] 08413X0091/F3 : Page 1 : 2505 résultats (2505/2505)
[15/60] 07475X0008/F3 : Page 1 : 2493 résultats (2493/2493)
[16/60] 07476X0034/PZF3 : Page 1 : 152

In [8]:
# 4c. Relance des stations en échec (une seule fois, délai d'attente triplé)

if erreurs:
    a_relancer = [e["code_bss"] for e in erreurs]
    print(f"Nouvelle tentative pour {len(a_relancer)} station(s)\n")
    mesures_relance, erreurs = recuperer_chroniques(a_relancer, timeout_s=TIMEOUT_S * 3)
    chroniques.extend(mesures_relance)

print(f"\n✅ Total : {len(chroniques)} mesures")
print(f"⚠️ {len(erreurs)} station(s) toujours en échec")
erreurs     # liste vide = tout s'est bien passé

Nouvelle tentative pour 4 station(s)

[1/4] 08426X0133/P2 : Page 1 : 2498 résultats (2498/2498)
[2/4] 08651X0038/0038 : Page 1 : 1758 résultats (1758/1758)
[3/4] 08191X0022/P : Page 1 : 2502 résultats (2502/2502)
[4/4] 08672X0008/HY : Page 1 : 2383 résultats (2383/2383)

✅ Total : 114291 mesures
⚠️ 0 station(s) toujours en échec


[]

#### 5. Mise en DataFrame et contrôle qualité
   5a. Création du DataFrame et conversion des types
   5b. Contrôle qualité : valeurs manquantes, doublons
   5c. Couverture : nombre de mesures et période par station
   5d. Fiabilité des mesures : qualification, statut, mode d'obtention

In [9]:
# 5a. Création du DataFrame et conversion des types

df_chroniques = pd.DataFrame(chroniques)
df_chroniques["date_mesure"] = pd.to_datetime(df_chroniques["date_mesure"])

print("Dimensions (lignes, colonnes) :", df_chroniques.shape)
df_chroniques.info()

Dimensions (lignes, colonnes) : (114291, 16)
<class 'pandas.DataFrame'>
RangeIndex: 114291 entries, 0 to 114290
Data columns (total 16 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   code_bss            114291 non-null  str           
 1   bss_id              114291 non-null  str           
 2   urn_bss             114291 non-null  str           
 3   date_mesure         114291 non-null  datetime64[us]
 4   timestamp_mesure    114291 non-null  int64         
 5   niveau_nappe_eau    114291 non-null  float64       
 6   mode_obtention      114291 non-null  str           
 7   statut              114291 non-null  str           
 8   qualification       114291 non-null  str           
 9   code_continuite     114291 non-null  str           
 10  nom_continuite      114291 non-null  str           
 11  code_producteur     114159 non-null  str           
 12  nom_producteur      114159 non-null  str           


In [10]:
# 5b. Contrôle qualité : valeurs manquantes, doublons

# Valeurs manquantes par colonne (uniquement celles qui en ont)
manquants = df_chroniques.isna().sum()
print("Valeurs manquantes :\n", manquants[manquants > 0].sort_values(ascending=False))

# Doublons : une seule mesure attendue par station et par jour
doublons = df_chroniques.duplicated(subset=["code_bss", "date_mesure"]).sum()
print("\nDoublons (code_bss + date_mesure) :", doublons)

Valeurs manquantes :
 code_producteur    132
nom_producteur     132
dtype: int64

Doublons (code_bss + date_mesure) : 0


In [11]:
# 5c. Couverture : nombre de mesures et période couverte par station

couverture = (df_chroniques
              .groupby("code_bss")
              .agg(nb_mesures=("date_mesure", "size"),
                   premiere_mesure=("date_mesure", "min"),
                   derniere_mesure=("date_mesure", "max"))
              .sort_values("nb_mesures"))

stations_sans_mesure = sorted(set(codes_stations) - set(couverture.index))

print(f"{len(couverture)} stations avec des mesures sur la période")
print(f"{len(stations_sans_mesure)} station(s) sans aucune mesure sur la période :", stations_sans_mesure)

# Cohérence : les stations sans mesure doivent être celles arrêtées avant la période (étape 3)
inattendues = set(stations_sans_mesure) - set(arretees["code_bss"])
print("Stations sans mesure NON expliquées par un arrêt avant la période :", sorted(inattendues) or "aucune ✅")
print("\nNombre de mesures par station :")
print(couverture["nb_mesures"].describe().round(0))

couverture.head(10)   # les stations les moins fournies

52 stations avec des mesures sur la période
8 station(s) sans aucune mesure sur la période : ['07944X0048/S', '07944X0049/S', '07952X0027/F', '08184X0069/S', '08188X0054/F', '08413X0081/LAUZAS', '08662X0049/P', '08666X0161/PUITS']
Stations sans mesure NON expliquées par un arrêt avant la période : aucune ✅

Nombre de mesures par station :
count      52.0
mean     2198.0
std       674.0
min        71.0
25%      2388.0
50%      2500.0
75%      2505.0
max      2505.0
Name: nb_mesures, dtype: float64


,nb_mesures,premiere_mesure,derniere_mesure
code_bss,,,
BSS004GTLS/X,71,2025-08-05,2025-10-14
07476X0034/PZF3,152,2025-05-16,2025-10-14
08184X0066/DRAIN,167,2020-01-01,2020-06-15
BSS004NBZY/X,474,2024-04-19,2025-08-05
07965X0081/HY,877,2020-01-01,2022-09-17
BSS004GYJR/X,915,2023-03-15,2026-04-24
07965X0092/F,1713,2021-06-25,2026-10-14
08651X0038/0038,1758,2021-12-14,2026-11-08
BSS004BKNJ/X,1846,2021-10-14,2026-11-08


In [12]:
# 5d. Fiabilité des mesures : répartition des valeurs (en % des mesures)

for colonne in ["qualification", "statut", "mode_obtention"]:
    print(f"\n--- {colonne} ---")
    print((df_chroniques[colonne].value_counts(dropna=False, normalize=True) * 100).round(1))


--- qualification ---
qualification
Correcte        97.9
Non qualifié     1.3
Incertaine       0.8
Incorrecte       0.0
Name: proportion, dtype: float64

--- statut ---
statut
Donnée contrôlée niveau 2    73.5
Donnée contrôlée niveau 1    24.7
Donnée brute                  1.3
Donnée interprétée            0.5
Name: proportion, dtype: float64

--- mode_obtention ---
mode_obtention
Valeur mesurée              95.3
Mode d'obtention inconnu     3.4
Valeur reconstituée          1.3
Name: proportion, dtype: float64


**Constat (extraction du 10/11/2026, période depuis le 01/01/2020) :** *(à mettre à jour si les chiffres changent)*
- **~114 000 mesures** pour **52 stations** sur 60 ; les **8 stations sans mesure** sont exactement celles arrêtées avant 2020 (étape 3) → cohérent.
- Une mesure par jour au maximum (environ 2 500 mesures pour une station complète depuis 2020) ; **aucun doublon** station + date.
- Quelques stations ne couvrent qu'une partie de la période (mises en service récentes ou arrêtées) : à prendre en compte avant de comparer des stations entre elles.
- Seules valeurs manquantes : le producteur de la donnée (`code_producteur`, `nom_producteur`) sur ~130 mesures → sans impact sur les niveaux.
- **Fiabilité** : ~98 % des mesures qualifiées « Correcte », ~98 % contrôlées (niveau 1 ou 2) ; les mesures « Incertaine » / « Incorrecte » / « Non qualifié » pourront être exclues lors de l'analyse.
- Les codes `BSS…/X` sont un nouveau format d'identifiant BSS : ils sont identiques dans les stations et les chroniques, la jointure fonctionne.

#### 6. Sauvegarde des données brutes

Données de l'API **telles quelles** + contexte de l'extraction (traçabilité).
Fichier volumineux → enregistré **sans indentation** pour limiter sa taille.

In [13]:
# 6. Sauvegarde des données brutes (data/raw/)

date_extraction = datetime.now()
fichier_brut = DATA_RAW / f"chroniques_{date_extraction:%Y-%m-%d}.json"

contenu_brut = {
    "date_extraction": date_extraction.isoformat(timespec="seconds"),
    "url": URL_CHRONIQUES,
    "params": {                                  # paramètres communs à tous les appels
        "date_debut_mesure": DATE_DEBUT_CHRONIQUES,
        "size": PAGE_SIZE_CHRONIQUES,
        "format": "json",
    },
    "stations_interrogees": codes_stations,
    "erreurs": erreurs,
    "nb_resultats": len(chroniques),
    "data": chroniques,
}

with open(fichier_brut, "w", encoding="utf-8") as f:
    json.dump(contenu_brut, f, ensure_ascii=False)     # sans indent : fichier plus léger

taille_mo = fichier_brut.stat().st_size / 1_000_000
print(f"✅ {len(chroniques)} mesures sauvegardées dans : {fichier_brut} ({taille_mo:.1f} Mo)")

✅ 114291 mesures sauvegardées dans : /home/virginie/Documents/DATA/00-PROJETS/Hub-eau/data/raw/chroniques_2026-11-10.json (66.8 Mo)


In [14]:
# Vérification : relecture du fichier sauvegardé
with open(fichier_brut, encoding="utf-8") as f:
    relu = json.load(f)

print("Date d'extraction :", relu["date_extraction"])
print("Nombre de mesures :", len(relu["data"]))
assert len(relu["data"]) == len(chroniques), "⚠️ Le fichier relu ne correspond pas aux données récupérées"

Date d'extraction : 2026-11-10T18:07:14
Nombre de mesures : 114291


#### 7. Sauvegarde des données traitées

On ne garde que les colonnes utiles à l'analyse (les autres restent disponibles dans le fichier brut).
Format **long** : 1 ligne = 1 station × 1 date. Le nom du fichier indique la période couverte.

In [15]:
# 7. Sauvegarde des données traitées (data/processed/)

COLONNES_UTILES = [
    "code_bss",            # identifiant de la station (clé de jointure avec les stations)
    "date_mesure",
    "niveau_nappe_eau",    # altitude de la surface de l'eau (m NGF)
    "profondeur_nappe",    # profondeur de l'eau sous le sol (m)
    "qualification",       # fiabilité de la mesure
    "statut",              # niveau de contrôle de la donnée
    "mode_obtention",
]

df_chroniques_propre = (df_chroniques[COLONNES_UTILES]
                        .sort_values(["code_bss", "date_mesure"])
                        .reset_index(drop=True))

fichier_traite = DATA_PROCESSED / f"chroniques_depuis_{DATE_DEBUT_CHRONIQUES}.csv"
df_chroniques_propre.to_csv(fichier_traite, index=False)

print(f"✅ {len(df_chroniques_propre)} mesures sauvegardées dans : {fichier_traite}")
df_chroniques_propre.head()

✅ 114291 mesures sauvegardées dans : /home/virginie/Documents/DATA/00-PROJETS/Hub-eau/data/processed/chroniques_depuis_2020-01-01.csv


,code_bss,date_mesure,niveau_nappe_eau,profondeur_nappe,qualification,statut,mode_obtention
0,07475X0008/F3,2020-01-01,288.03,22.15,Correcte,Donnée contrôlée niveau 2,Valeur mesurée
1,07475X0008/F3,2020-01-02,288.01,22.17,Correcte,Donnée contrôlée niveau 2,Valeur mesurée
2,07475X0008/F3,2020-01-03,288.00,22.18,Correcte,Donnée contrôlée niveau 2,Valeur mesurée
3,07475X0008/F3,2020-01-04,287.99,22.19,Correcte,Donnée contrôlée niveau 2,Valeur mesurée
4,07475X0008/F3,2020-01-05,287.99,22.19,Correcte,Donnée contrôlée niveau 2,Valeur mesurée
